# 01 — base data (`data/basedata/*.tsv`)

`data/fitness_browser/feba.db`(2024-02 스냅샷)에서 **세균 46종** 의 단백질코딩 유전자(`Gene.type`=1) 전량을 뽑아, 각 유전자 결실주의 **무자극(대조) 조건 적합도(fitness)** 를 대표값 2종 · 결합 통계량 2종 · 원시 리스트 3종으로 정리해 균주별 TSV 46개(`순번_org_id.tsv`, 순번 = `NOSTRESS` 순서)로 저장한다. 파일을 순번대로 이어 붙인 행 순서가 02 노트북 임베딩 h5 의 행 순서다.

## 유전자 3분류

| 분류 | 판정 | `fit_*` | `comb_*` | `list_*` |
|---|---|---|---|---|
| **필수** | 필수목록에 있음 | `-inf` | 공란 | 공란 |
| **비필수(measured)** | 무자극 실험에 쓸 수 있는 fit 값이 있음 | 대표값 2종 | 결합 표준오차·유사 $t$ | fit·표준오차·유사 $t$ 리스트 |
| **미상(unknown)** | 위 어느 쪽도 아님 | 공란 | 공란 | 공란 |

세 분류가 코딩 유전자를 정확히 분할하므로 결과 TSV에는 균주별 프로테옴 전량이 한 번씩 들어간다.

## 열 구성 (13개)

`org_id` · `org_name` · `gene_id` · `gene_name` · `seq_dna` · `seq_aa` · `fit_mean` · `fit_median` · `comb_se` · `comb_t` · `list_fit` · `list_se` · `list_t`

- `gene_id` = `Gene.locusId`, `gene_name` = `Gene.sysName`(비었으면 `Gene.gene`)
- `seq_dna` = 게놈 서열에서 잘라낸 유전자 서열(역방향 가닥은 역상보), `seq_aa` = `aaseqs`의 단백질 서열
- `fit_*` 대표값 2종은 3.2절, `comb_se`·`comb_t`는 3.3절
- `list_t` = `GeneFitness.t` 원값 ($t = fit / se$)
- 세 `list_*` 는 같은 마스크·같은 순서를 쓰므로 원소가 일대일 대응한다
- 모든 수치는 소숫점 6자리 고정

In [1]:
# !pip install biopython pandas openpyxl

In [2]:
import os, sqlite3, warnings, numpy as np, pandas as pd
from Bio import SeqIO
from Bio.Seq import reverse_complement

warnings.filterwarnings('ignore')
DB      = '../data/fitness_browser/feba.db'
REFER   = '../data/refer/'               # 필수 유전자 목록 보충표 (저장소에 포함)
ESS_TAB = '../data/fitness_browser/essential_proteins.tab'
AASEQS  = '../data/fitness_browser/aaseqs'
OUT_DIR = '../data/basedata'              # 균주별 TSV '순번_org_id.tsv' 저장 폴더
DEC     = 6                       # fit 계열 소숫점 자리수

FIT_COLS  = ['fit_mean', 'fit_median']
COMB_COLS = ['comb_se', 'comb_t']
LIST_COLS = ['list_fit', 'list_se', 'list_t']
COLS      = ['org_id', 'org_name', 'gene_id', 'gene_name', 'seq_dna', 'seq_aa'] \
            + FIT_COLS + COMB_COLS + LIST_COLS

con = sqlite3.connect(DB)

## 1. 입력 — 균주별 무자극 실험 목록

In [3]:
NOSTRESS = {
    'acidovorax_3H11'           : ['set3IT059', 'set3IT060'],
    'ANA3'                      : ['set4IT075'],  # ★다른 배양온도
    'azobra'                    : ['set1IT052', 'set1IT060'],  # ★다른 배양온도
    'BFirm'                     : ['set2IT067'],
    'Btheta'                    : ['set1IT071', 'set1IT072', 'set4IT047', 'set4IT050'],  # ‡다른 배지 §정의배지
    'Burk376'                   : ['set1IT043', 'set1IT044', 'set4IT079', 'set4IT088'],  # ‡다른 배지 §정의배지
    'Caulo'                     : ['set8IT011', 'set8IT023', 'set8IT035'],
    'Cola'                      : ['set1IT035', 'set1IT048', 'set1IT079', 'set2IT011', 'set2IT024', 'set2IT034', 'set2IT045', 'set2IT058', 'set2IT068'],
    'Cup4G11'                   : ['set4IT069', 'set5IT004', 'set5IT012'],
    'Dda3937'                   : ['set9TCH417', 'set9TCH418', 'set9TCH419', 'set9TCH420', 'setATCH110', 'setATCH111', 'setATCH112', 'setATCH191', 'setATCH192'],  # ★다른 배양온도
    'Ddia6719'                  : ['setATCH203', 'setATCH204', 'setATCH206', 'setATCH207'],  # ★다른 배양온도
    'DdiaME23'                  : ['set10TCH436', 'set10TCH438', 'setATCH113', 'setATCH114', 'setATCH115', 'setATCH240', 'setATCH241', 'setATCH242'],  # ★다른 배양온도
    'Dino'                      : ['set7IT086', 'set7IT087'],  # †품질 이탈 제외
    'DvH'                       : ['set1S209', 'set1S210', 'set1S211', 'set2S268', 'set2S269', 'set2S270', 'set3S344', 'set3S345', 'set3S346', 'set4IT040', 'set4IT041', 'set4IT042'],
    'Dyella79'                  : ['set3IT040', 'set3IT051', 'set3IT060', 'set3IT067'],
    'HerbieS'                   : ['set12IT038', 'set12IT039', 'set12IT040', 'set13IT052', 'set13IT053', 'set13IT054'],  # ‡다른 배지 ‖다른 라이브러리
    'Kang'                      : ['set2H11', 'set2H20', 'set2H43', 'set2H48'],  # †품질 이탈 제외
    'Keio'                      : ['set2IT045', 'set2IT069'],  # ★다른 배양온도
    'Korea'                     : ['set5IT075', 'set5IT076'],
    'Koxy'                      : ['set6IT088', 'set7IT016', 'set7IT054', 'set7IT071', 'set7IT078'],
    'Magneto'                   : ['set1IT053'],
    'Marino'                    : ['set2H22', 'set2H31', 'set2H41', 'set3H25', 'set3H30', 'set3H31', 'set3H39', 'set3H7', 'set4H10', 'set4H18', 'set4H30'],
    'Miya'                      : ['set2IT012', 'set2IT016', 'set2IT025'],
    'MR1'                       : ['set1H15', 'set2H29', 'set2H44', 'set4H43', 'set4H48', 'set5H12', 'set5H44', 'set5H47', 'set6H11', 'set6H18', 'set6H25', 'set6H7'],
    'Pedo557'                   : ['set1IT019', 'set1IT034', 'set1IT053', 'set1IT068', 'set1IT095', 'set2IT008', 'set2IT022', 'set2IT032', 'set2IT041'],
    'Phaeo'                     : ['set2H31', 'set2H43', 'set2H5', 'set3H7', 'set4H18', 'set6IT009', 'set6IT016', 'set6IT030', 'set6IT041', 'set7IT053', 'set7IT062', 'set7IT073', 'set7IT085', 'set8IT088', 'set8IT096'],
    'Ponti'                     : ['set1IT035', 'set1IT045', 'set1IT063', 'set1IT073', 'set2IT084', 'set2IT096', 'set3IT015', 'set3IT028', 'set3IT040', 'set3IT055'],
    'PS'                        : ['set1H13', 'set1H15', 'set1H16'],
    'pseudo1_N1B4'              : ['set5IT018'],
    'pseudo3_N2E3'              : ['set6IT053', 'set6IT054'],
    'pseudo5_N2C3_1'            : ['set6IT074', 'set6IT075'],
    'pseudo6_N2E2'              : ['set3IT072', 'set3IT076', 'set3IT082', 'set3IT085'],
    'pseudo13_GW456_L13'        : ['set2IT087'],  # ★다른 배양온도
    'psRCH2'                    : ['set3H43', 'set4H19', 'set4H37', 'set4H6', 'set5H14', 'set5H5'],
    'Putida'                    : ['set36IT074', 'set36IT075', 'set36IT076', 'set38IT007', 'set38IT010', 'set38IT011'],  # ‡다른 배지 ‖다른 라이브러리
    'PV4'                       : ['set1H1', 'set1H25', 'set1H6', 'set3IT055'],
    'RalstoniaBSBF1503'         : ['set1IT076', 'set1IT080', 'set3IT039'],
    'RalstoniaGMI1000'          : ['set2IT062', 'set4IT033'],
    'RalstoniaPSI07'            : ['set1IT085', 'set1IT089'],
    'RalstoniaUW163'            : ['set1IT067', 'set1IT071', 'set2IT036'],
    'SB2B'                      : ['set1H42', 'set1H43', 'set1H44', 'set2H12', 'set2H6', 'set2H7', 'set4H10', 'set4H20', 'set4H34', 'set4H42', 'set5H11', 'set5H23', 'set5H7', 'setAH41'],
    'Smeli'                     : ['set2IT069', 'set2IT070'],
    'SynE'                      : ['set3IT004', 'set3IT016', 'set3IT031', 'set4IT050', 'set4IT062', 'set4IT077', 'set4IT089', 'set7IT032', 'set7IT042', 'set7IT047', 'set8IT052', 'set8IT057', 'set8IT062', 'set8IT066', 'set8IT069', 'set8IT080', 'set8IT085', 'set8IT090'],
    'SyringaeB728a'             : ['set1IT005', 'set4IT013'],
    'SyringaeB728a_mexBdelta'   : ['set1IT006', 'set4IT014'],
    'WCS417'                    : ['set18IT044'],  # ‡다른 배지
}
print(f'organisms {len(NOSTRESS)} | no-stress experiments {sum(map(len, NOSTRESS.values()))}')

organisms 46 | no-stress experiments 226


## 2. 필수 유전자 목록 — 6개 소스

| 소스 | 파일 | 필수 판정 열 | 균주 |
|---|---|---|---|
| Price18 계열 | `essential_proteins.tab` | (파일 전체가 필수목록) | 32종 |
| Liu21 | `Liu21_Btheta_essential.xlsx` `TableS1` | (시트 전체가 필수목록) | Btheta |
| Trotter21 | `Trotter21_DvH_essential.xlsx` `SupplementaryTable 5` | `ess.wt` | DvH |
| Helmann19 | `Helmann19_SyringaeB728a_essential.xlsx` `Dataset S1` | `ess` | SyringaeB728a 2종 |
| Helmann21 | `Helmann21_{orgId}_essential.csv` | `ess` | Dickeya 3종 |
| Georgoulis21 | `Georgoulis21_ST1_essential.xlsx` 균주별 시트 | `Predicted essential by Essentiality.pl script` | Ralstonia 3종 |

유전자 키는 6종 모두 `locusId`다. 나머지 4종(Burk376·Magneto·Putida·RalstoniaUW163)은 필수성 자료가 없어 필수 판정이 0건이 된다.

- `essential_proteins.tab`은 orgId가 아니라 **학명** 으로 균주를 식별하므로 `Organism`(genus+species+strain)과 맞춘다. Cup4G11은 표기가 달라 수동 매핑한다.
- 보충표 엑셀은 헤더 행 위치가 파일마다 다르므로 **비어있지 않은 셀이 5개 이상인 첫 행** 을 헤더로 잡는다.

In [4]:
ORGNAME_FIX = {'Cupriavidus basilensis 4G11': 'Cup4G11'}     # 학명 표기가 다른 1건
name2org = {f'{g} {s} {st}'.strip(): o for o, g, s, st in
            con.execute('SELECT orgId,genus,species,strain FROM Organism')}

def xl(fname, sheet, min_cols=5):
    """전문(preamble)이 있는 보충표 시트 -> 실제 데이터 표 DataFrame"""
    raw = pd.read_excel(REFER + fname, sheet_name=sheet, header=None)
    hdr = next(i for i in range(len(raw)) if raw.iloc[i].notna().sum() >= min_cols)
    return pd.read_excel(REFER + fname, sheet_name=sheet, header=hdr)

def is_true(s):
    """TRUE/True/yes/1 등 표기 차이를 흡수한 불리언 마스크"""
    return s.astype(str).str.strip().str.lower().isin(['true', 'yes', '1'])

# (1) Price18 계열 32종 - 학명으로 orgId 매핑
tab = pd.read_csv(ESS_TAB, sep='\t', usecols=['organism', 'locusId'], dtype=str)
tab['orgId'] = tab.organism.map(lambda s: ORGNAME_FIX.get(s) or name2org.get(s))
ESS = {o: set(d.locusId) for o, d in tab.groupby('orgId')}

# (2) 저자 보고 목록 10종
d = xl('Liu21_Btheta_essential.xlsx', 'TableS1')
ESS['Btheta'] = set(d.locusId.astype(str))                                   # 시트 전체가 필수목록
d = xl('Trotter21_DvH_essential.xlsx', 'SupplementaryTable 5')
ESS['DvH'] = set(d.locusId.astype(str)[is_true(d['ess.wt'])])                 # 야생형 기준(ess.wt)
d = xl('Helmann19_SyringaeB728a_essential.xlsx', 'Dataset S1')
ESS['SyringaeB728a'] = ESS['SyringaeB728a_mexBdelta'] = \
    set(d.locusId.astype(str)[is_true(d['ess'])])                             # B728a 게놈 공유
for org in ['Dda3937', 'Ddia6719', 'DdiaME23']:
    d = pd.read_csv(f'{REFER}Helmann21_{org}_essential.csv', dtype=str)
    ESS[org] = set(d.locusId[is_true(d['ess'])])
PL_COL = 'Predicted essential by Essentiality.pl script'
for org, sheet in [('RalstoniaGMI1000', 'GMI1000'), ('RalstoniaBSBF1503', 'IBSBF1503'),
                   ('RalstoniaPSI07', 'PSI07')]:
    d = xl('Georgoulis21_ST1_essential.xlsx', sheet)
    ESS[org] = set(d.locusId.astype(str)[is_true(d[PL_COL])])

ESS = {o: s for o, s in ESS.items() if o in NOSTRESS}                          # 대상 46종만
print(f'essential genes {sum(map(len, ESS.values())):,} over {len(ESS)} organisms')

essential genes 18,037 over 42 organisms


## 3. 대표값 2종 · 결합 통계량 2종 · 표준오차

### 3.1 표준오차 — 유사 $t$ 통계량에서 역산

feba.db는 실험별로 적합도 `fit`(log2 비)과 유사 $t$ 통계량 `t`를 준다. $t_i = fit_i / se_i$ 이므로

$$se_i = \frac{\lvert fit_i \rvert}{\lvert t_i \rvert}$$

`fit`=0인 값은 `t`도 정확히 0이어서 $se$ 가 정의되지 않는다. 이런 값은 리스트·대표값·결합값 계산에서 모두 제외한다.

### 3.2 두 가지 대표값

$n$ 개 무자극 실험의 $fit_i$ 로부터,

| 열 | 정의 |
|---|---|
| `fit_mean` | $\frac{1}{n}\sum_i fit_i$ |
| `fit_median` | $\mathrm{median}(fit_i)$ |

### 3.3 결합 통계량 — `comb_se` · `comb_t`

무자극 실험 $n$ 개는 같은 조건의 반복이지만, `Experiment.timeZeroSet` 이 같은 실험은 시작 시료를 공유해 서로 독립이 아니다. 이 열로 실험을 $G$ 개 그룹으로 나눈 뒤 **그룹 안은 공유 보정, 그룹 사이는 독립 결합** 을 적용한다.

잡음 모형은 실험 $i$ 가 그룹 $g$ 에 속할 때 $fit_i = f - s_g + e_i$ 이며, 그룹 공유 성분 $s_g$ 와 실험별 독립 성분 $e_i$ 가 $se_i^2$ 의 절반씩을 차지한다.

$$t_g = \frac{\sum_{i \in g} t_i}{\sqrt{n_g(n_g+1)/2}}, \qquad comb\_t = \frac{\sum_g t_g}{\sqrt{G}}$$

$$comb\_se = \sqrt{\frac{1}{2n^2}\sum_g (n_g+1)\,S_g}, \qquad S_g = \sum_{i \in g} se_i^2$$

$n_g$ 는 그룹 $g$ 의 유효 실험 수, $G$ 는 유효 실험이 하나 이상 있는 그룹 수다.

| 열 | 짝 | 쓰임 |
|---|---|---|
| `comb_t` | 특정 대표값에 매이지 않음 | 무자극 적합도가 0과 다른가의 **유의성**. 판정 기준 $\lvert comb\_t \rvert > 4$ |
| `comb_se` | **`fit_mean` 전용** | `fit_mean` 의 표준오차. 손실 가중치·불확실도 전파·유전자 간 차이 검정 |

`comb_se` 는 `fit_median` 에는 쓰지 않는다. 유전자 두 개의 결합 적합도 비교는 차이를 따로 검정한다.

$$z = \frac{fit\_mean_A - fit\_mean_B}{\sqrt{comb\_se_A^2 + comb\_se_B^2}}$$

feba.db의 실험별 $se_i$ 에는 하한 $\sigma = 0.1$ 이 들어 있으나, 결합식은 독립 성분을 평균으로 줄이므로 $comb\_se$ 는 0.1 아래로 내려갈 수 있다(measured 177,317개 중 30.0%). **바닥을 적용하지 않고 원값을 그대로 저장** 하므로, 필요하면 쓰는 쪽에서 $\max(comb\_se,\ 0.1)$ 로 둔다.

> **추정한 부분** — Price18이 직접 제시한 것은 단일 그룹에 대한 $comb\_t$ 두 형태(독립·공유)뿐이다. 위 $comb\_t$ 는 그 두 공식을 Time0 그룹 구조에 2단으로 적용한 것이고, $comb\_se$ 는 같은 잡음 가정에서 전개한 것이다. 그룹 안의 $se_i$ 가 서로 비슷하다고 보므로 편차가 큰 그룹에서는 근사다.

In [5]:
def fit_stats(F, ok):
    """
    F  : [n_gene, n_exp] 적합도 (무효 칸은 NaN)
    ok : [n_gene, n_exp] 유효 마스크
    반환 : [2, n_gene] = mean / median (유효 실험 0개인 유전자는 NaN)
    """
    has = ok.any(1)                                       # [n_gene] 유효 실험 1개 이상
    out = np.full((2, len(F)), np.nan)
    out[0, has] = np.nanmean(F[has], axis=1)              # mean
    out[1, has] = np.nanmedian(F[has], axis=1)            # median
    return out


def combine_reps(SE, T, ok, gid, n_group):
    """
    SE, T   : [n_gene, n_exp] 표준오차·유사 t (무효 칸은 NaN)
    ok      : [n_gene, n_exp] 유효 마스크
    gid     : [n_exp] 실험별 Time0 집합 번호 (같은 번호 = 같은 시작 시료 공유)
    n_group : Time0 집합 개수
    반환    : (comb_se [n_gene], comb_t [n_gene]) - 유효 실험 0개인 유전자는 NaN
    """
    okf = ok.astype(float)
    n   = ok.sum(1)
    M   = np.zeros((len(gid), n_group))                    # one-hot [n_exp, n_group]
    M[np.arange(len(gid)), gid] = 1.0

    ng = okf @ M                                           # [n_gene, n_group] 그룹별 유효 실험 수
    Sg = np.where(ok, SE, 0.0) ** 2 @ M                    # [n_gene, n_group] 그룹별 se^2 합
    Tg = np.where(ok, T, 0.0) @ M                          # [n_gene, n_group] 그룹별 t 합

    with np.errstate(invalid='ignore', divide='ignore'):
        comb_se = np.sqrt(((ng + 1) * Sg).sum(1) / (2 * n ** 2))          # fit_mean의 표준오차
        tg      = np.where(ng > 0, Tg / np.sqrt(ng * (ng + 1) / 2), 0.0)  # 그룹 안 - 공유 보정
        comb_t  = tg.sum(1) / np.sqrt((ng > 0).sum(1))                    # 그룹 사이 - Stouffer
    comb_se[n == 0] = comb_t[n == 0] = np.nan
    return comb_se, comb_t


def fmt_lists(ok, mats):
    """
    ok   : [n_gene, n_exp] 유효 마스크
    mats : 같은 모양의 실수 행렬 묶음 (fit, se, t 순으로 넘긴다)
    반환 : 행렬 수만큼의 문자열 리스트 - 유효 값이 없는 유전자는 공란
    """
    any_ok = ok.any(1)
    out    = []
    for M in mats:
        S = np.char.mod(f'%.{DEC}f', np.where(ok, M, 0.0))   # 소숫점 DEC자리 문자열화
        out.append(['[' + ', '.join(s[m]) + ']' if a else ''
                    for s, m, a in zip(S, ok, any_ok)])
    return out

## 4. 서열 — `seq_dna` · `seq_aa`

- `seq_dna` — `ScaffoldSeq.sequence`(균주 게놈 서열)를 `Gene.begin`~`Gene.end`(1-기반 양끝 포함)로 자르고, `Gene.strand`가 `-`면 역상보(reverse complement). 종결코돈을 포함한다.
- `seq_aa` — `aaseqs`(FASTA, 헤더 `orgId:locusId`). 코딩 유전자 전용 파일이다.

In [6]:
# aaseqs 전량 적재 (헤더 'orgId:locusId') -> {org: {locusId: seq}}
AA = {}
for rec in SeqIO.parse(AASEQS, 'fasta'):
    org, _, lid = rec.id.partition(':')
    AA.setdefault(org, {})[lid] = str(rec.seq)

def gene_dna(genes, scaffolds):
    """
    genes     : Gene 행(scaffoldId/begin/end/strand)
    scaffolds : {scaffoldId: 게놈 서열}
    반환      : 유전자 DNA 서열 리스트 (역가닥은 역상보, 종결코돈 포함)
    """
    out = []
    for c, b, e, st in zip(genes.scaffoldId.to_numpy(), genes.begin.to_numpy(),
                           genes.end.to_numpy(), genes.strand.to_numpy()):
        seq = scaffolds[c][b - 1:e]                    # 1-기반 양끝 포함 -> 0-기반 슬라이스
        out.append(reverse_complement(seq) if st == '-' else seq)
    return out

## 5. 조립 및 저장 — 균주별 TSV

균주별로 (코딩 유전자 표) × (무자극 실험 목록) 밀집 행렬을 만들어 통계를 벡터화 계산한다. 유전자 순서는 게놈 좌표 순(`scaffoldId`, `begin`)이다.

각 균주에서 `fit`·`t` 는 **피벗 1회** 로 함께 뽑고, 무효 칸 마스크 `ok` 를 한 번 만들어 세 행렬(`F`·`SE`·`T`)과 대표값·결합값·리스트 계산에 공유한다. `Experiment.timeZeroSet` 은 루프 밖에서 한 번만 조회해 균주별 Time0 집합 번호로 factorize 한다.

필수목록에 있으면서 무자극 fit도 있는 유전자 22건은 **필수 판정을 우선** 해 `fit_*` 를 `-inf`, 나머지를 공란으로 덮는다.

In [7]:
NEG   = float('-inf')
NOSET = {o: set(v) for o, v in NOSTRESS.items()}

# Time0 집합 - 실험별 timeZeroSet {org: {expName: tz}}
TZ = {}
for o, e, z in con.execute('SELECT orgId,expName,timeZeroSet FROM Experiment'):
    if o in NOSET and e in NOSET[o]:
        TZ.setdefault(o, {})[e] = z

report, n_drop, n_conflict = [], 0, 0
os.makedirs(OUT_DIR, exist_ok=True)

for i, (org, exps) in enumerate(NOSTRESS.items()):
    org_name = ' '.join(con.execute('SELECT genus,species,strain FROM Organism WHERE orgId=?',
                                    (org,)).fetchone()).strip()
    # (1) 코딩 유전자 표 - 게놈 좌표 순
    genes = pd.read_sql('SELECT locusId,sysName,gene,scaffoldId,begin,end,strand FROM Gene '
                        'WHERE orgId=? AND type=1 ORDER BY scaffoldId,begin', con, params=(org,))

    # (2) 무자극 실험 fit/t -> [n_gene, n_exp] 밀집 행렬 (피벗 1회, 입력 실험 순서 유지)
    ph  = ','.join('?' * len(exps))
    raw = pd.read_sql(f'SELECT locusId,expName,fit,t FROM FitByExp_{org} WHERE expName IN ({ph})',
                      con, params=exps)
    piv = raw.pivot(index='locusId', columns='expName', values=['fit', 't']) \
             .reindex(index=genes.locusId)
    F = piv['fit'].reindex(columns=exps).to_numpy(float)
    T = piv['t'].reindex(columns=exps).to_numpy(float)

    # (3) 표준오차 se = |fit|/|t| ; t=0(=fit 0)은 se 정의 불가 -> 세 행렬에서 함께 제외
    with np.errstate(invalid='ignore', divide='ignore'):
        SE = np.abs(F) / np.abs(T)
    n_drop += int((np.isfinite(T) & (T == 0)).sum())
    ok      = np.isfinite(F) & np.isfinite(SE)
    F, SE, T = (np.where(ok, M, np.nan) for M in (F, SE, T))
    n_ok    = ok.sum(1)

    # (4) 대표값 2종 + 결합 통계량 + 원시 리스트 (Time0 집합 구조 반영)
    gid, tzs = pd.factorize(np.asarray([TZ[org][e] for e in exps], dtype=object))
    stats            = fit_stats(F, ok)
    comb_se, comb_t  = combine_reps(SE, T, ok, gid, len(tzs))
    l_fit, l_se, l_t = fmt_lists(ok, (F, SE, T))

    # (5) 필수 여부 (필수인데 fit도 있는 충돌은 필수 우선)
    is_ess      = np.isin(genes.locusId.to_numpy(), list(ESS.get(org, ())))
    n_conflict += int((is_ess & (n_ok > 0)).sum())

    # (6) 서열
    scaffolds = dict(con.execute('SELECT scaffoldId,sequence FROM ScaffoldSeq WHERE orgId=?', (org,)))
    aa_org    = AA.get(org, {})
    sysname   = genes.sysName.fillna('')

    out = pd.DataFrame({
        'org_id':     org,
        'org_name':   org_name,
        'gene_id':    genes.locusId,
        'gene_name':  sysname.where(sysname != '', genes.gene.fillna('')),   # sysName 없으면 gene
        'seq_dna':    gene_dna(genes, scaffolds),
        'seq_aa':     [aa_org.get(lid, '') for lid in genes.locusId],
        'fit_mean':   stats[0], 'fit_median': stats[1],
        'comb_se':    comb_se,  'comb_t':     comb_t,
        'list_fit':   l_fit,    'list_se':    l_se,   'list_t': l_t})

    # (7) 필수 -> fit 전부 -inf, 결합값·리스트는 공란 / 미상 -> 전부 공란(NaN 그대로)
    out.loc[is_ess, FIT_COLS]  = NEG
    out.loc[is_ess, COMB_COLS] = np.nan
    out.loc[is_ess, LIST_COLS] = ''

    out[COLS].to_csv(os.path.join(OUT_DIR, f'{i+1:02d}_{org}.tsv'), sep='\t', index=False,   # 균주별 파일 (순번 = NOSTRESS 순서)
                     float_format=f'%.{DEC}f', na_rep='', lineterminator='\n')
    report.append(dict(org_id=org, n_exp=len(exps), n_tz=len(tzs), rows=len(out),
                       essential=int(is_ess.sum()),
                       measured=int((~is_ess & (n_ok > 0)).sum()),
                       unknown=int((~is_ess & (n_ok == 0)).sum()),
                       aa_len=int(out.seq_aa.str.len().sum())))

report = pd.DataFrame(report)
print(report.drop(columns='aa_len').to_string(index=False))
print(f'\nsaved {OUT_DIR}/ ({len(report)} files)')
print(f'organisms {len(report)} | rows {report.rows.sum():,} = essential {report.essential.sum():,} '
      f'+ measured {report.measured.sum():,} + unknown {report.unknown.sum():,}')
print(f'residues {report.aa_len.sum():,} | dropped values (t=0) {n_drop} | '
      f'essential-measured conflicts {n_conflict}')

                 org_id  n_exp  n_tz  rows  essential  measured  unknown
        acidovorax_3H11      2     1  4964        504      3924      536
                   ANA3      1     1  4360        307      3642      411
                 azobra      2     1  5488        340      4178      970
                  BFirm      1     1  7182        409      5421     1352
                 Btheta      4     2  4816        378      4054      384
                Burk376      4     2  6555          0      5147     1408
                  Caulo      3     1  3886        440      3305      141
                   Cola      9     2  4625        492      3950      183
                Cup4G11      3     2  7358        474      6373      511
                Dda3937      9     3  4213        374      3609      230
               Ddia6719      4     1  4110        426      3429      255
               DdiaME23      8     3  4182        426      3606      150
                   Dino      2     1  4192        5